# Day 3：手写 FlashAttention 与推理优化 — 从标准 Attention 到 Tiling + Online Softmax

> **目标**：从零实现标准 Attention 并分析其 IO 开销；手写 FlashAttention 前向传播（Python 级 tiling + Online Softmax）；通过数值对比验证正确性；使用 PyTorch `scaled_dot_product_attention` 进行 benchmark；实现带 KV Cache 和 GQA 的推理优化。

---

## 总体流程

```
Part 1: 标准 Attention 实现 + IO 计数
  标准公式实现 → 显存分析 → IO 计数验证

Part 2: Online Softmax 实现
  标准 Softmax → 分块 Softmax → Online 更新验证

Part 3: 手写 FlashAttention 前向传播
  Tiling + Online Softmax → Python 级实现

Part 4: 正确性验证
  与标准 Attention 对比 → 数值误差分析

Part 5: PyTorch SDPA Benchmark
  scaled_dot_product_attention → 后端对比 → 性能分析

Part 6: KV Cache + GQA 推理优化
  KV Cache 实现 → GQA 推理 → 吞吐量对比
```

In [1]:
from __future__ import annotations

import math
import time
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")

Device: cuda
GPU: NVIDIA GeForce RTX 4090


## Part 1：标准 Attention 实现 + IO 计数

### 1.1 标准 Self-Attention

$$O = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

我们先实现一个标准版本，并统计 HBM 读写次数作为 baseline。

In [2]:
def standard_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    """
    标准 Scaled Dot-Product Attention.
    Q, K, V: (batch, n_heads, seq_len, d_head)
    """
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, N, N)
    attn_weights = torch.softmax(scores, dim=-1)                     # (B, H, N, N)
    output = torch.matmul(attn_weights, V)                           # (B, H, N, d)
    return output


def count_standard_attention_io(N: int, d: int, bytes_per_elem: int = 2) -> dict:
    """
    统计标准 Attention 的 HBM 读写量 (bytes).
    假设 Q, K, V, O 存在 HBM，中间矩阵 S, P 也写入 HBM.
    """
    read_qkv = 3 * N * d * bytes_per_elem
    write_s = N * N * bytes_per_elem
    read_s_for_softmax = N * N * bytes_per_elem
    write_p = N * N * bytes_per_elem
    read_p_and_v = N * N * bytes_per_elem + N * d * bytes_per_elem
    write_o = N * d * bytes_per_elem

    total_read = read_qkv + read_s_for_softmax + read_p_and_v
    total_write = write_s + write_p + write_o

    return {
        "total_read_bytes": total_read,
        "total_write_bytes": total_write,
        "total_io_bytes": total_read + total_write,
        "n_squared_terms_bytes": 3 * N * N * bytes_per_elem,
        "n_d_terms_bytes": 4 * N * d * bytes_per_elem,
    }


# 测试标准 Attention
B, H, N, d = 1, 1, 1024, 128
Q = torch.randn(B, H, N, d, device=device, dtype=torch.float32)
K = torch.randn(B, H, N, d, device=device, dtype=torch.float32)
V = torch.randn(B, H, N, d, device=device, dtype=torch.float32)

output_std = standard_attention(Q, K, V)
print(f"Output shape: {output_std.shape}")

# IO 分析
for seq_len in [512, 1024, 4096, 8192]:
    io = count_standard_attention_io(seq_len, 128)
    n2_pct = io["n_squared_terms_bytes"] / io["total_io_bytes"] * 100
    print(
        f"N={seq_len:5d}: total IO = {io['total_io_bytes'] / 1e6:8.2f} MB, "
        f"N² terms = {n2_pct:.1f}%"
    )

Output shape: torch.Size([1, 1, 1024, 128])
N=  512: total IO =     2.75 MB, N² terms = 57.1%
N= 1024: total IO =     9.70 MB, N² terms = 64.9%
N= 4096: total IO =   139.46 MB, N² terms = 72.2%
N= 8192: total IO =   547.36 MB, N² terms = 73.6%


### 1.2 观察

当 $N$ 增大时，$O(N^2)$ 的中间矩阵 $S$ 和 $P$ 占据了绝大部分 IO。
这就是 FlashAttention 要消除的瓶颈——**永远不把 $N \times N$ 矩阵写入 HBM**。

---

## Part 2：Online Softmax 实现

### 2.1 标准 Softmax 的问题

标准 softmax 需要两遍扫描：
1. 第一遍：求 $m = \max(x)$ 和 $\ell = \sum e^{x_i - m}$
2. 第二遍：计算 $\text{softmax}(x_i) = e^{x_i - m} / \ell$

如果数据分块到来，每来一个新块就需要重新遍历所有之前的块。

### 2.2 Online Softmax

Online Softmax 的核心：当新块到来时，用标量修正因子更新之前的结果。

$$m_{\text{new}} = \max(m_{\text{old}}, m_{\text{block}})$$
$$\ell_{\text{new}} = \ell_{\text{old}} \cdot e^{m_{\text{old}} - m_{\text{new}}} + \ell_{\text{block}} \cdot e^{m_{\text{block}} - m_{\text{new}}}$$

In [3]:
def standard_softmax(x: torch.Tensor) -> torch.Tensor:
    """数值稳定的标准 softmax (单遍不分块)"""
    m = x.max(dim=-1, keepdim=True).values
    exp_x = torch.exp(x - m)
    return exp_x / exp_x.sum(dim=-1, keepdim=True)


def online_softmax(x: torch.Tensor, block_size: int) -> torch.Tensor:
    """
    分块计算 softmax (Online Softmax).
    x: (N,) 一维向量
    block_size: 每块大小
    """
    N = x.shape[0]
    n_blocks = math.ceil(N / block_size)

    m = torch.tensor(float("-inf"), device=x.device)
    l = torch.tensor(0.0, device=x.device)

    exp_values = torch.zeros(N, device=x.device)

    for j in range(n_blocks):
        start = j * block_size
        end = min(start + block_size, N)
        x_block = x[start:end]

        m_block = x_block.max()
        m_new = torch.max(m, m_block)

        # 修正之前所有块的 exp 值
        exp_values[:start] *= torch.exp(m - m_new)

        # 计算当前块的 exp 值
        exp_values[start:end] = torch.exp(x_block - m_new)

        # 更新统计量
        l = l * torch.exp(m - m_new) + exp_values[start:end].sum()
        m = m_new

    return exp_values / l


# 验证 Online Softmax 的正确性
x = torch.randn(256, device=device)

result_std = standard_softmax(x)
result_online = online_softmax(x, block_size=32)

max_diff = (result_std - result_online).abs().max().item()
print(f"Standard vs Online Softmax max diff: {max_diff:.2e}")
assert max_diff < 1e-5, "Online Softmax 结果不正确!"
print("✓ Online Softmax 验证通过!")

Standard vs Online Softmax max diff: 3.73e-09
✓ Online Softmax 验证通过!


### 2.3 带 Value 加权的 Online Softmax

在 Attention 中，softmax 的结果需要乘以 $V$：

$$o = \sum_j \text{softmax}(s)_j \cdot v_j$$

分块时，我们需要同时维护未归一化的加权和：

In [4]:
def online_softmax_weighted(scores: torch.Tensor, V: torch.Tensor,
                            block_size: int) -> torch.Tensor:
    """
    分块计算 softmax(scores) @ V，使用 Online Softmax.
    scores: (N,) — 单行的注意力分数
    V: (N, d) — Value 矩阵
    block_size: KV 块大小
    """
    N, d = V.shape
    n_blocks = math.ceil(N / block_size)

    m = torch.tensor(float("-inf"), device=scores.device)
    l = torch.tensor(0.0, device=scores.device)
    o = torch.zeros(d, device=scores.device)

    for j in range(n_blocks):
        start = j * block_size
        end = min(start + block_size, N)

        s_block = scores[start:end]          # (block,)
        v_block = V[start:end]               # (block, d)

        m_block = s_block.max()
        m_new = torch.max(m, m_block)

        # 修正因子
        correction_old = torch.exp(m - m_new)
        correction_new = torch.exp(m_block - m_new)

        p_block = torch.exp(s_block - m_new)  # (block,)
        l_block = p_block.sum()

        # 更新输出: 修正旧的 + 加入新的
        o = correction_old * o + p_block @ v_block  # (d,)
        l = correction_old * l + l_block
        m = m_new

    return o / l


# 验证
N_test, d_test = 256, 64
scores_test = torch.randn(N_test, device=device)
V_test = torch.randn(N_test, d_test, device=device)

# 标准方法
attn_weights = torch.softmax(scores_test, dim=0)
output_standard = attn_weights @ V_test  # (d,)

# Online 方法
output_online = online_softmax_weighted(scores_test, V_test, block_size=32)

max_diff = (output_standard - output_online).abs().max().item()
print(f"Standard vs Online weighted softmax max diff: {max_diff:.2e}")
assert max_diff < 1e-4, "带 V 加权的 Online Softmax 结果不正确!"
print("✓ 带 Value 加权的 Online Softmax 验证通过!")

Standard vs Online weighted softmax max diff: 8.94e-08
✓ 带 Value 加权的 Online Softmax 验证通过!


---

## Part 3：手写 FlashAttention 前向传播

### 3.1 核心思想回顾

FlashAttention 将 Q, K, V 分成小块，在 SRAM 中完成注意力计算，
永远不把 $N \times N$ 矩阵写入 HBM。

这里我们用 Python 实现 FA 的逻辑（不涉及 CUDA kernel），
核心是 **tiling + Online Softmax**。

In [ ]:
def flash_attention_forward(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    block_size_q: int = 64,
    block_size_kv: int = 64,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    FlashAttention 前向传播 (Python 级实现).

    采用 FA2 的循环顺序: 外循环遍历 Q 块, 内循环遍历 KV 块.

    Q, K, V: (batch, n_heads, seq_len, d_head)
    返回: (O, l, m) — 输出, 行 log-sum-exp, 行最大值
    """
    B, H, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    O = torch.zeros_like(Q)                                # (B, H, N, d)
    l = torch.zeros(B, H, N, 1, device=Q.device, dtype=Q.dtype)  # 行归一化因子
    m = torch.full((B, H, N, 1), float("-inf"), device=Q.device, dtype=Q.dtype)  # 行最大值

    n_blocks_q = math.ceil(N / block_size_q)
    n_blocks_kv = math.ceil(N / block_size_kv)

    for i in range(n_blocks_q):  # 外循环: Q 块
        q_start = i * block_size_q
        q_end = min(q_start + block_size_q, N)

        Q_block = Q[:, :, q_start:q_end, :]     # (B, H, Bq, d)
        O_block = O[:, :, q_start:q_end, :]     # (B, H, Bq, d)
        l_block = l[:, :, q_start:q_end, :]     # (B, H, Bq, 1)
        m_block = m[:, :, q_start:q_end, :]     # (B, H, Bq, 1)

        for j in range(n_blocks_kv):  # 内循环: KV 块
            kv_start = j * block_size_kv
            kv_end = min(kv_start + block_size_kv, N)

            K_block = K[:, :, kv_start:kv_end, :]  # (B, H, Bkv, d)
            V_block = V[:, :, kv_start:kv_end, :]  # (B, H, Bkv, d)

            # 计算局部注意力分数
            S_block = torch.matmul(Q_block, K_block.transpose(-2, -1)) * scale
            # S_block: (B, H, Bq, Bkv)

            # 局部统计量
            m_block_new = S_block.max(dim=-1, keepdim=True).values  # (B, H, Bq, 1)
            m_new = torch.max(m_block, m_block_new)

            # 修正因子
            correction_old = torch.exp(m_block - m_new)
            correction_new = torch.exp(m_block_new - m_new)

            # 局部 softmax 分子
            P_block = torch.exp(S_block - m_new)  # (B, H, Bq, Bkv)
            l_block_new = P_block.sum(dim=-1, keepdim=True)  # (B, H, Bq, 1)

            # Online 更新输出
            # O_block: (B, H, Bq, d)
            # correction_old: (B, H, Bq, 1)
            # torch.matmul(P_block, V_block): (B, H, Bq, d)
            O_block = correction_old * O_block + torch.matmul(P_block, V_block)  # (B, H, Bq, d)

            # 更新统计量
            l_block = correction_old * l_block + l_block_new
            m_block = m_new

        # 最终归一化
        O_block = O_block / l_block

        # 写回
        O[:, :, q_start:q_end, :] = O_block
        l[:, :, q_start:q_end, :] = l_block
        m[:, :, q_start:q_end, :] = m_block

    return O, l, m


# 快速测试
B, H, N, d = 1, 2, 256, 64
Q = torch.randn(B, H, N, d, device=device)
K = torch.randn(B, H, N, d, device=device)
V = torch.randn(B, H, N, d, device=device)

O_flash, _, _ = flash_attention_forward(Q, K, V, block_size_q=64, block_size_kv=64)
print(f"FlashAttention output shape: {O_flash.shape}")

FlashAttention output shape: torch.Size([1, 2, 256, 64])


---

## Part 4：正确性验证

将手写 FlashAttention 的输出与标准 Attention 对比。

In [6]:
def verify_flash_attention(
    batch_size: int = 2,
    n_heads: int = 4,
    seq_len: int = 512,
    d_head: int = 64,
    block_size_q: int = 64,
    block_size_kv: int = 64,
):
    """对比标准 Attention 和手写 FlashAttention 的输出"""
    Q = torch.randn(batch_size, n_heads, seq_len, d_head, device=device)
    K = torch.randn(batch_size, n_heads, seq_len, d_head, device=device)
    V = torch.randn(batch_size, n_heads, seq_len, d_head, device=device)

    # 标准 Attention
    O_std = standard_attention(Q, K, V)

    # 手写 FlashAttention
    O_flash, _, _ = flash_attention_forward(Q, K, V, block_size_q, block_size_kv)

    # 对比
    max_diff = (O_std - O_flash).abs().max().item()
    mean_diff = (O_std - O_flash).abs().mean().item()
    cos_sim = F.cosine_similarity(O_std.flatten(), O_flash.flatten(), dim=0).item()

    print(f"Config: B={batch_size}, H={n_heads}, N={seq_len}, d={d_head}, "
          f"Bq={block_size_q}, Bkv={block_size_kv}")
    print(f"  Max  absolute diff: {max_diff:.2e}")
    print(f"  Mean absolute diff: {mean_diff:.2e}")
    print(f"  Cosine similarity:  {cos_sim:.10f}")
    print(f"  {'✓ PASS' if max_diff < 1e-3 else '✗ FAIL'}")
    print()


print("=" * 60)
print("FlashAttention 正确性验证")
print("=" * 60)

# 不同配置下验证
verify_flash_attention(batch_size=1, n_heads=1, seq_len=128, d_head=64)
verify_flash_attention(batch_size=2, n_heads=4, seq_len=256, d_head=64)
verify_flash_attention(batch_size=1, n_heads=8, seq_len=512, d_head=128)

# 不同 block size
verify_flash_attention(batch_size=1, n_heads=4, seq_len=256, d_head=64,
                       block_size_q=32, block_size_kv=32)
verify_flash_attention(batch_size=1, n_heads=4, seq_len=256, d_head=64,
                       block_size_q=128, block_size_kv=64)

# 非整除情况
verify_flash_attention(batch_size=1, n_heads=2, seq_len=300, d_head=64,
                       block_size_q=64, block_size_kv=64)

FlashAttention 正确性验证
Config: B=1, H=1, N=128, d=64, Bq=64, Bkv=64
  Max  absolute diff: 7.75e-07
  Mean absolute diff: 3.83e-08
  Cosine similarity:  1.0000000000
  ✓ PASS

Config: B=2, H=4, N=256, d=64, Bq=64, Bkv=64
  Max  absolute diff: 4.17e-07
  Mean absolute diff: 2.42e-08
  Cosine similarity:  1.0000000000
  ✓ PASS

Config: B=1, H=8, N=512, d=128, Bq=64, Bkv=64
  Max  absolute diff: 5.07e-07
  Mean absolute diff: 2.30e-08
  Cosine similarity:  1.0000000000
  ✓ PASS

Config: B=1, H=4, N=256, d=64, Bq=32, Bkv=32
  Max  absolute diff: 3.43e-07
  Mean absolute diff: 2.32e-08
  Cosine similarity:  0.9999999404
  ✓ PASS

Config: B=1, H=4, N=256, d=64, Bq=128, Bkv=64
  Max  absolute diff: 3.28e-07
  Mean absolute diff: 2.46e-08
  Cosine similarity:  1.0000000000
  ✓ PASS

Config: B=1, H=2, N=300, d=64, Bq=64, Bkv=64
  Max  absolute diff: 2.98e-07
  Mean absolute diff: 2.34e-08
  Cosine similarity:  1.0000000000
  ✓ PASS



---

## Part 5：PyTorch SDPA Benchmark

### 5.1 使用 `scaled_dot_product_attention`

PyTorch 2.0+ 提供 `F.scaled_dot_product_attention`，自动选择后端（FlashAttention-2 / Memory-Efficient / Math）。

In [7]:
def benchmark_attention(
    batch_size: int,
    n_heads: int,
    seq_len: int,
    d_head: int,
    n_warmup: int = 5,
    n_iters: int = 20,
):
    """对比不同 Attention 实现的速度"""
    Q = torch.randn(batch_size, n_heads, seq_len, d_head,
                    device=device, dtype=torch.float16)
    K = torch.randn(batch_size, n_heads, seq_len, d_head,
                    device=device, dtype=torch.float16)
    V = torch.randn(batch_size, n_heads, seq_len, d_head,
                    device=device, dtype=torch.float16)

    results = {}

    # 1. PyTorch SDPA (auto backend)
    for _ in range(n_warmup):
        _ = F.scaled_dot_product_attention(Q, K, V)
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(n_iters):
        _ = F.scaled_dot_product_attention(Q, K, V)
    torch.cuda.synchronize()
    results["SDPA (auto)"] = (time.perf_counter() - start) / n_iters * 1000

    # 2. 标准 Attention (math backend)
    try:
        from torch.nn.attention import SDPBackend, sdpa_kernel

        with sdpa_kernel(SDPBackend.MATH):
            for _ in range(n_warmup):
                _ = F.scaled_dot_product_attention(Q, K, V)
            torch.cuda.synchronize()

            start = time.perf_counter()
            for _ in range(n_iters):
                _ = F.scaled_dot_product_attention(Q, K, V)
            torch.cuda.synchronize()
            results["SDPA (math)"] = (time.perf_counter() - start) / n_iters * 1000
    except Exception:
        pass

    # 3. 手写 FlashAttention (Python 级, FP32)
    Q32, K32, V32 = Q.float(), K.float(), V.float()
    for _ in range(n_warmup):
        _ = flash_attention_forward(Q32, K32, V32)
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(n_iters):
        _ = flash_attention_forward(Q32, K32, V32)
    torch.cuda.synchronize()
    results["Flash (Python)"] = (time.perf_counter() - start) / n_iters * 1000

    print(f"\nConfig: B={batch_size}, H={n_heads}, N={seq_len}, d={d_head}")
    print("-" * 50)
    for name, ms in results.items():
        print(f"  {name:20s}: {ms:8.2f} ms")

    if "SDPA (math)" in results and "SDPA (auto)" in results:
        speedup = results["SDPA (math)"] / results["SDPA (auto)"]
        print(f"  SDPA auto speedup vs math: {speedup:.1f}×")


if device == "cuda":
    print("=" * 60)
    print("Attention Benchmark")
    print("=" * 60)
    benchmark_attention(1, 32, 512, 128)
    benchmark_attention(1, 32, 2048, 128)
    benchmark_attention(1, 32, 4096, 128)
else:
    print("需要 CUDA GPU 才能运行 benchmark")

Attention Benchmark

Config: B=1, H=32, N=512, d=128
--------------------------------------------------
  SDPA (auto)         :     0.07 ms
  SDPA (math)         :     0.32 ms
  Flash (Python)      :    17.50 ms
  SDPA auto speedup vs math: 4.9×

Config: B=1, H=32, N=2048, d=128
--------------------------------------------------
  SDPA (auto)         :     0.46 ms
  SDPA (math)         :     5.87 ms
  Flash (Python)      :   148.74 ms
  SDPA auto speedup vs math: 12.8×

Config: B=1, H=32, N=4096, d=128
--------------------------------------------------
  SDPA (auto)         :     1.81 ms
  SDPA (math)         :    23.21 ms
  Flash (Python)      :   688.06 ms
  SDPA auto speedup vs math: 12.8×


### 5.2 Benchmark 分析

| 观察 | 原因 |
|------|------|
| SDPA auto 远快于 math | auto 后端使用 FlashAttention-2 CUDA kernel |
| Python 版 Flash 很慢 | Python 循环开销大，真正的 FA 是 CUDA kernel |
| 序列越长加速越大 | FA 将 IO 从 $O(N^2)$ 降到 $O(N^2 d / M)$ |

我们手写的 Python 版本 **验证了算法正确性**，但速度不是重点。
真正的加速来自 Tri Dao 的 CUDA 实现。

---

## Part 6：KV Cache + GQA 推理优化

### 6.1 KV Cache 实现

在 Decode 阶段，每步只有 1 个新 token 的 Q，需要与缓存的所有 K, V 做 Attention。

In [8]:
class KVCache:
    """简单的 KV Cache 实现"""

    def __init__(self, max_seq_len: int, n_kv_heads: int, d_head: int,
                 device: str = "cuda", dtype: torch.dtype = torch.float16):
        self.max_seq_len = max_seq_len
        self.cache_k = torch.zeros(1, n_kv_heads, max_seq_len, d_head,
                                   device=device, dtype=dtype)
        self.cache_v = torch.zeros(1, n_kv_heads, max_seq_len, d_head,
                                   device=device, dtype=dtype)
        self.seq_len = 0

    def update(self, k: torch.Tensor, v: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        追加新的 KV 并返回完整缓存.
        k, v: (1, n_kv_heads, new_len, d_head)
        """
        new_len = k.shape[2]
        self.cache_k[:, :, self.seq_len:self.seq_len + new_len, :] = k
        self.cache_v[:, :, self.seq_len:self.seq_len + new_len, :] = v
        self.seq_len += new_len

        return (
            self.cache_k[:, :, :self.seq_len, :],
            self.cache_v[:, :, :self.seq_len, :],
        )

    def memory_bytes(self) -> int:
        """当前已用显存 (bytes)"""
        elem_size = self.cache_k.element_size()
        return 2 * self.seq_len * self.cache_k.shape[1] * self.cache_k.shape[3] * elem_size

    def reset(self):
        self.seq_len = 0


# 演示 KV Cache
n_kv_heads, d_head, max_len = 8, 128, 2048
cache = KVCache(max_len, n_kv_heads, d_head, device=device)

print("KV Cache 演示:")
# Prefill: 一次写入 100 tokens
k_prefill = torch.randn(1, n_kv_heads, 100, d_head, device=device, dtype=torch.float16)
v_prefill = torch.randn(1, n_kv_heads, 100, d_head, device=device, dtype=torch.float16)
full_k, full_v = cache.update(k_prefill, v_prefill)
print(f"  After prefill: cache_len={cache.seq_len}, K shape={full_k.shape}, "
      f"memory={cache.memory_bytes() / 1024:.1f} KB")

# Decode: 逐 token 追加
for step in range(5):
    k_new = torch.randn(1, n_kv_heads, 1, d_head, device=device, dtype=torch.float16)
    v_new = torch.randn(1, n_kv_heads, 1, d_head, device=device, dtype=torch.float16)
    full_k, full_v = cache.update(k_new, v_new)

print(f"  After 5 decode steps: cache_len={cache.seq_len}, K shape={full_k.shape}, "
      f"memory={cache.memory_bytes() / 1024:.1f} KB")

KV Cache 演示:
  After prefill: cache_len=100, K shape=torch.Size([1, 8, 100, 128]), memory=400.0 KB
  After 5 decode steps: cache_len=105, K shape=torch.Size([1, 8, 105, 128]), memory=420.0 KB


### 6.2 GQA Attention 推理

GQA 中 Query 头数 > KV 头数，需要将 KV 头 `repeat_interleave` 到与 Query 对齐。

In [9]:
def gqa_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    n_heads: int,
    n_kv_heads: int,
) -> torch.Tensor:
    """
    Grouped-Query Attention.
    Q: (batch, n_heads, seq_q, d_head)
    K: (batch, n_kv_heads, seq_kv, d_head)
    V: (batch, n_kv_heads, seq_kv, d_head)
    """
    group_size = n_heads // n_kv_heads

    K_expanded = K.repeat_interleave(group_size, dim=1)  # (batch, n_heads, seq_kv, d_head)
    V_expanded = V.repeat_interleave(group_size, dim=1)

    return F.scaled_dot_product_attention(Q, K_expanded, V_expanded)


# 演示 GQA
n_heads, n_kv_heads, d_head = 32, 8, 128
seq_q, seq_kv = 1, 512

Q = torch.randn(1, n_heads, seq_q, d_head, device=device, dtype=torch.float16)
K = torch.randn(1, n_kv_heads, seq_kv, d_head, device=device, dtype=torch.float16)
V = torch.randn(1, n_kv_heads, seq_kv, d_head, device=device, dtype=torch.float16)

output = gqa_attention(Q, K, V, n_heads, n_kv_heads)
print(f"GQA output shape: {output.shape}")
print(f"Q heads: {n_heads}, KV heads: {n_kv_heads}, group size: {n_heads // n_kv_heads}")
print(f"KV Cache 节省: {1 - n_kv_heads / n_heads:.0%}")

GQA output shape: torch.Size([1, 32, 1, 128])
Q heads: 32, KV heads: 8, group size: 4
KV Cache 节省: 75%


### 6.3 完整推理循环：KV Cache + GQA

In [10]:
class SimpleGQALayer(nn.Module):
    """简化的 GQA 注意力层 (用于演示 KV Cache 推理)"""

    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.d_head = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.wq = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.wo = nn.Linear(n_heads * self.d_head, d_model, bias=False)

    def forward(self, x: torch.Tensor,
                kv_cache: Optional[KVCache] = None) -> torch.Tensor:
        B, T, _ = x.shape

        q = self.wq(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if kv_cache is not None:
            k, v = kv_cache.update(k, v)

        k_expanded = k.repeat_interleave(self.group_size, dim=1)
        v_expanded = v.repeat_interleave(self.group_size, dim=1)

        attn_out = F.scaled_dot_product_attention(q, k_expanded, v_expanded)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, -1)

        return self.wo(attn_out)


# 演示推理循环
d_model, n_heads, n_kv_heads = 512, 8, 2
layer = SimpleGQALayer(d_model, n_heads, n_kv_heads).to(device).half()
d_head = d_model // n_heads

# 不使用 KV Cache (每步重新计算)
prompt_len, gen_len = 50, 20
tokens = torch.randn(1, prompt_len, d_model, device=device, dtype=torch.float16)

start = time.perf_counter()
for step in range(gen_len):
    x_input = torch.cat([tokens, torch.randn(1, 1, d_model, device=device, dtype=torch.float16)], dim=1)
    out = layer(x_input)
    tokens = x_input
torch.cuda.synchronize()
time_no_cache = (time.perf_counter() - start) * 1000

# 使用 KV Cache
cache = KVCache(prompt_len + gen_len + 10, n_kv_heads, d_head, device=device)
tokens_cached = torch.randn(1, prompt_len, d_model, device=device, dtype=torch.float16)

start = time.perf_counter()
# Prefill
out = layer(tokens_cached, kv_cache=cache)
# Decode
for step in range(gen_len):
    x_step = torch.randn(1, 1, d_model, device=device, dtype=torch.float16)
    out = layer(x_step, kv_cache=cache)
torch.cuda.synchronize()
time_with_cache = (time.perf_counter() - start) * 1000

print(f"\n推理性能对比 (prompt={prompt_len}, gen={gen_len}):")
print(f"  无 KV Cache: {time_no_cache:.2f} ms")
print(f"  有 KV Cache: {time_with_cache:.2f} ms")
print(f"  加速比:      {time_no_cache / time_with_cache:.1f}×")
print(f"  KV Cache 显存: {cache.memory_bytes() / 1024:.1f} KB")


推理性能对比 (prompt=50, gen=20):
  无 KV Cache: 587.52 ms
  有 KV Cache: 86.01 ms
  加速比:      6.8×
  KV Cache 显存: 35.0 KB


### 6.4 KV Cache 显存分析工具

In [11]:
def analyze_kv_cache_memory(
    model_name: str,
    n_layers: int,
    n_kv_heads: int,
    d_head: int,
    seq_lengths: list[int],
    batch_sizes: list[int],
    dtype_bytes: int = 2,
):
    """分析不同配置下的 KV Cache 显存"""
    print(f"\n{'=' * 70}")
    print(f"KV Cache 显存分析: {model_name}")
    print(f"  Layers={n_layers}, KV_heads={n_kv_heads}, d_head={d_head}, "
          f"dtype={'FP16' if dtype_bytes == 2 else 'FP32'}")
    print(f"{'=' * 70}")

    header = f"{'Seq Len':>10}"
    for bs in batch_sizes:
        header += f" | BS={bs:>3}"
    print(header)
    print("-" * len(header))

    for seq_len in seq_lengths:
        row = f"{seq_len:>10}"
        for bs in batch_sizes:
            mem = 2 * n_layers * n_kv_heads * d_head * seq_len * bs * dtype_bytes
            mem_gb = mem / (1024 ** 3)
            row += f" | {mem_gb:>6.2f} GB"
        print(row)


# 分析主流模型
seq_lengths = [512, 2048, 4096, 8192, 32768]
batch_sizes = [1, 4, 16, 64]

analyze_kv_cache_memory("LLaMA-2-7B (MHA)",
                        n_layers=32, n_kv_heads=32, d_head=128,
                        seq_lengths=seq_lengths, batch_sizes=batch_sizes)

analyze_kv_cache_memory("LLaMA-3-8B (GQA-8)",
                        n_layers=32, n_kv_heads=8, d_head=128,
                        seq_lengths=seq_lengths, batch_sizes=batch_sizes)

analyze_kv_cache_memory("LLaMA-2-70B (GQA-8)",
                        n_layers=80, n_kv_heads=8, d_head=128,
                        seq_lengths=seq_lengths, batch_sizes=batch_sizes)


KV Cache 显存分析: LLaMA-2-7B (MHA)
  Layers=32, KV_heads=32, d_head=128, dtype=FP16
   Seq Len | BS=  1 | BS=  4 | BS= 16 | BS= 64
----------------------------------------------
       512 |   0.25 GB |   1.00 GB |   4.00 GB |  16.00 GB
      2048 |   1.00 GB |   4.00 GB |  16.00 GB |  64.00 GB
      4096 |   2.00 GB |   8.00 GB |  32.00 GB | 128.00 GB
      8192 |   4.00 GB |  16.00 GB |  64.00 GB | 256.00 GB
     32768 |  16.00 GB |  64.00 GB | 256.00 GB | 1024.00 GB

KV Cache 显存分析: LLaMA-3-8B (GQA-8)
  Layers=32, KV_heads=8, d_head=128, dtype=FP16
   Seq Len | BS=  1 | BS=  4 | BS= 16 | BS= 64
----------------------------------------------
       512 |   0.06 GB |   0.25 GB |   1.00 GB |   4.00 GB
      2048 |   0.25 GB |   1.00 GB |   4.00 GB |  16.00 GB
      4096 |   0.50 GB |   2.00 GB |   8.00 GB |  32.00 GB
      8192 |   1.00 GB |   4.00 GB |  16.00 GB |  64.00 GB
     32768 |   4.00 GB |  16.00 GB |  64.00 GB | 256.00 GB

KV Cache 显存分析: LLaMA-2-70B (GQA-8)
  Layers=80, KV_head

---

## 总结与面试要点

### 本日核心收获

| 知识点 | 掌握程度 |
|--------|----------|
| 标准 Attention IO 瓶颈 | 能定量分析 $O(N^2)$ IO |
| Online Softmax | 理解分块更新公式，能手写 |
| FlashAttention 前向 | 能写 Python 级实现并验证正确性 |
| PyTorch SDPA | 知道如何使用和选择后端 |
| KV Cache | 能实现基本 cache 并分析显存 |
| GQA 推理 | 理解 repeat_interleave 和显存节省 |

### 面试高频

1. **手写 FlashAttention 前向伪代码** — 今天已经实现了 Python 版本
2. **Online Softmax 的更新规则** — Part 2 中验证过
3. **KV Cache 显存计算** — Part 6.4 的分析工具
4. **GQA 如何减少 KV Cache** — Part 6.2 的 repeat_interleave

### 与后续内容的衔接

```
Day 3: 手写 FlashAttention + KV Cache + GQA
  → Day 4: PagedAttention 解决 KV Cache 碎片化
  → Day 5-6: 量化进一步压缩模型和 KV Cache
```